# 06 - Predictive Model: Transaction Sales Value

Train an XGBoost regressor to predict transaction-level `sales` value from
country, channel, sub-channel, product class, sales team, manager, month, year, and quantity.
Use SHAP to interpret what drives sales value - tying together findings from
segmentation (country/segment differences) and rep performance (team differences).

In [ ]:
import sys
sys.path.append("../src")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from eda_utils import load_clean_data, save_fig
from segmentation import filter_valid_sales, add_order_type
from model import (
    build_model_dataset, train_xgb_model, evaluate_model,
    plot_actual_vs_predicted, plot_feature_importance,
    compute_shap_values, plot_shap_summary, plot_shap_bar,
    decode_category, CATEGORICAL_FEATURES, NUMERIC_FEATURES
)

In [ ]:
df = load_clean_data("../data/processed/cleaned_data.csv")
df = filter_valid_sales(df)
df = add_order_type(df, bulk_quantile=0.99)
print(f"Dataset size: {len(df):,} rows")
df.head()

## 1. Build model dataset

Note: `quantity` is included as a feature, but it's highly correlated with `sales` (sales ~= quantity * price,
and price isn't in our feature set). This makes quantity a very strong predictor almost by construction.
We'll check both with and without quantity to see how much it dominates.

In [ ]:
X, y, encoder, feature_names = build_model_dataset(df)
print(f"Features: {feature_names}")
print(f"X shape: {X.shape}, y shape: {y.shape}")
X.head()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## 2. Train model (with quantity)

In [ ]:
model = train_xgb_model(X_train, y_train)
metrics = evaluate_model(model, X_test, y_test)
print({k: round(v, 2) for k, v in metrics.items() if k != 'predictions'})

In [ ]:
fig = plot_actual_vs_predicted(y_test, metrics['predictions'])
save_fig(fig, '23_actual_vs_predicted', path='../reports/figures')
plt.show()

In [ ]:
fig = plot_feature_importance(model, feature_names)
save_fig(fig, '24_feature_importance', path='../reports/figures')
plt.show()

## 3. Model without quantity

More useful for understanding which *business/territory factors* (not transaction size)
drive sales value - relevant for territory/segment-level recommendations.

In [ ]:
X2, y2, encoder2, feature_names2 = build_model_dataset(
    df,
    categorical_features=CATEGORICAL_FEATURES,
    numeric_features=['month_num', 'year']  # quantity excluded
)

X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size=0.2, random_state=42)

model2 = train_xgb_model(X2_train, y2_train)
metrics2 = evaluate_model(model2, X2_test, y2_test)
print({k: round(v, 2) for k, v in metrics2.items() if k != 'predictions'})

In [ ]:
fig = plot_feature_importance(model2, feature_names2)
save_fig(fig, '25_feature_importance_no_quantity', path='../reports/figures')
plt.show()

## 4. SHAP analysis (no-quantity model)

Using a sample for speed - SHAP on the full 254K-row dataset would be slow.

In [ ]:
sample_size = 2000
X2_sample = X2_test.sample(n=min(sample_size, len(X2_test)), random_state=42)

explainer, shap_values = compute_shap_values(model2, X2_sample)

In [ ]:
fig = plot_shap_bar(shap_values, X2_sample)
save_fig(fig, '26_shap_bar', path='../reports/figures')
plt.show()

In [ ]:
fig = plot_shap_summary(shap_values, X2_sample)
save_fig(fig, '27_shap_summary', path='../reports/figures')
plt.show()

## 5. Decode top categories for the most important feature

If `country`, `sales_team`, or `product_class` ranks highly, decode which specific
category values push predictions up vs down.

In [ ]:
# Example: decode country categories (adjust 'feature' based on SHAP results above)
for feature in ['country', 'sales_team', 'product_class']:
    idx = feature_names2.index(feature)
    categories = encoder2.categories_[CATEGORICAL_FEATURES.index(feature)]
    print(f"{feature}: {dict(enumerate(categories))}")

## Key Observations (fill in after running)

- Model performance (R2, MAPE) with vs without quantity - how much does transaction size dominate?
- Top 3-5 drivers of sales value per SHAP (no-quantity model)
- Does country/segment rank as a top driver, consistent with earlier findings?
- Business recommendation: which factors (channel, product class, team) should sales strategy prioritize based on their impact on transaction value?